# SchoolPulse: AI-Powered Student Wellbeing Monitor
### Kaggle AI Agents Intensive 2026 · Agents for Good
*A privacy-first, multi-agent pipeline that gives school counselors a daily synthesized brief — without replacing their judgment.*

In [ ]:
# ── Cell 2 · Environment setup & LLM client initialisation ───────────────────
# Loads GOOGLE_API_KEY from .env (Jupyter kernels don't inherit shell exports,
# so we parse the file directly). Creates RealSignalLLM and RealOrchestratorLLM —
# the two Gemini-backed clients injected into the orchestrator for the live demo.
# Defines _fmt() / _short() date helpers shared across all display cells.
# To run offline without an API key, replace Real* with Fake* (tests do this).

import os
import sys
import time
import copy
import logging
from pathlib import Path
from datetime import datetime, timezone

# Suppress httpx and google.genai INFO logs (HTTP request traces, AFC messages)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("google.genai").setLevel(logging.WARNING)

# Notebook lives in notebook/; repo root is one level up
REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

# Load .env from repo root if GOOGLE_API_KEY is not already in the environment
_env_file = REPO_ROOT / '.env'
if _env_file.exists() and not os.environ.get('GOOGLE_API_KEY'):
    for _line in _env_file.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith('#') and '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k.strip(), _v.strip())

from agents.orchestrator import SchoolPulseOrchestrator, load_data, DEMO_DATES
from agents.llm_interface import RealSignalLLM, RealOrchestratorLLM

GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    raise EnvironmentError(
        'GOOGLE_API_KEY not set. '
        'Either export it in your shell before launching VS Code, '
        'or add it to school-pulse/.env as:  GOOGLE_API_KEY=your-key'
    )

signal_llm = RealSignalLLM(api_key=GOOGLE_API_KEY)
orchestrator_llm = RealOrchestratorLLM(api_key=GOOGLE_API_KEY)

# Date helpers used across cells
def _fmt(d):
    """'2026-06-28' -> 'June 28, 2026'"""
    dt = datetime.strptime(d, '%Y-%m-%d')
    return dt.strftime('%B ') + str(dt.day) + ', ' + str(dt.year)

def _short(d):
    """'2026-06-28' -> 'Jun 28'"""
    dt = datetime.strptime(d, '%Y-%m-%d')
    return dt.strftime('%b ') + str(dt.day)

print('✓ SchoolPulse ready. Model: gemini-3.1-flash-lite')

In [ ]:
# ── Cell 3 · Load synthetic dataset ──────────────────────────────────────────
# Reads the three JSON fixtures from data/synthetic/:
#   synthetic_checkins.json   — 140 check-in records (20 students × 7 days)
#   teacher_observations.json — teacher flag notes per student per day
#   student_registry.json     — student metadata: age_group, arc_label, fictional_name
# Prints a summary to confirm the dataset loaded cleanly before running the pipeline.

DATA_DIR = REPO_ROOT / 'data' / 'synthetic'
checkins, teacher_obs, registry = load_data(DATA_DIR)

n_junior = sum(1 for r in registry if r['age_group'] == 'junior')
n_senior = sum(1 for r in registry if r['age_group'] == 'senior')

first_day = datetime.strptime(DEMO_DATES[0], '%Y-%m-%d').day
last_day  = datetime.strptime(DEMO_DATES[-1], '%Y-%m-%d').day
month     = datetime.strptime(DEMO_DATES[0], '%Y-%m-%d').strftime('%B')
year      = datetime.strptime(DEMO_DATES[0], '%Y-%m-%d').year

print('Dataset loaded:')
print(f'  Students : {len(registry)} ({n_junior} junior · {n_senior} senior)')
print(f'  Days     : {len(DEMO_DATES)}  ({month} {first_day}–{last_day}, {year})')
print(f'  Check-ins: {len(checkins)}')
print(f'  Brief covers data through: {_fmt(DEMO_DATES[-1])}')

In [ ]:
# ── Cell 4 · MCP data layer — tool demonstration ─────────────────────────────
# In production, check-ins and teacher observations are ingested via the Google
# Sheets MCP server (official @modelcontextprotocol/server-gdrive, wired in
# mcp_config.json). In this demo, mcp_server.py serves the same JSON fixtures
# through identical MCP tool signatures (stdio transport, FastMCP).
#
# This cell calls the four MCP tools directly to show data flows through the
# protocol boundary before reaching Privacy Guard. The pipeline cells below
# use the same underlying data — here we're demonstrating the interface.
# Whitepaper principle: consumption over creation — production path consumes
# the official Google Sheets MCP rather than a bespoke REST wrapper.

import sys
sys.path.insert(0, str(REPO_ROOT))
import mcp_server as mcp_data

print('MCP server tools (schoolpulse-local, stdio transport):')
print()

dates = mcp_data.list_available_dates()
print(f'  list_available_dates()                   → {dates}')

sample_checkins = mcp_data.get_daily_checkins(DEMO_DATES[-1])
print(f'  get_daily_checkins({DEMO_DATES[-1]!r})  → {len(sample_checkins)} records')

sample_obs = mcp_data.get_teacher_observations(DEMO_DATES[-1])
has_teacher = any('teacher_name' in r for r in sample_obs)
print(f'  get_teacher_observations({DEMO_DATES[-1]!r}) → {len(sample_obs)} records'
      f'  (teacher_name present: {has_teacher})')

reg = mcp_data.get_student_registry()
print(f'  get_student_registry()                   → {len(reg)} students')

print()
print('Production: copy mcp_config.json → ~/.gemini/config/mcp_config.json')
print('  "google-sheets-mcp" entry uses @modelcontextprotocol/server-gdrive')

## Architecture

```
                    ┌──────────────┐
                    │  MCP LAYER   │
                    │  Google      │
                    │  Sheets:     │
                    │  check-ins + │
                    │  teacher obs │
                    └──────┬───────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                     ORCHESTRATOR AGENT                          │
│              (coordinates pipeline, owns HITL gate)             │
└───────────────────────┬─────────────────────────────────────────┘
                        │
          ┌─────────────┼──────────────────┐
          ▼             ▼                  ▼
  ┌───────────────┐ ┌──────────────┐ ┌──────────────────┐
  │ PRIVACY       │ │ SIGNAL       │ │ MEMORY           │
  │ GUARD         │ │ DETECTOR     │ │ KEEPER           │
  │ Sub-Agent     │ │ Sub-Agent    │ │ Sub-Agent        │
  └───────┬───────┘ └──────┬───────┘ └────────┬─────────┘
          │                │                   │
   pii-context-    emotional-          student-trend-
   sanitizer       signal-reader       tracker
          │                │                   │
          └────────────────┼───────────────────┘
                           │
                    ┌──────▼───────┐
                    │ LLM-AS-JUDGE │
                    │ EVAL LAYER   │
                    └──────────────┘
                           │
                    ┌──────▼───────┐
                    │  COUNSELOR   │
                    │  DAILY BRIEF │
                    │  (HITL gate) │
                    └──────────────┘
```

**MCP Layer** — Data enters here. Check-ins and teacher observations from Google Sheets; local JSON fixtures (`data/synthetic/`) in this demo.

**Orchestrator Agent** — Runs the pipeline in three phases: Privacy Guard for all students, one batched LLM call for all senior signals, then Signal Detector and Memory Keeper per student. Assembles the Daily Brief, invokes the LLM-as-judge, and owns the HITL gate.

**Privacy Guard** — First in the pipeline: strips all PII before any data enters an LLM context window.

**Signal Detector** — Converts check-ins to structured emotional signals. Junior students: emoji lookup table. Senior students: gemini-3.1-flash-lite text parse (all seniors batched into one LLM call per day).

**Memory Keeper** — Maintains a 7-day rolling window per student, tracks the valence baseline, and fires pattern-break detection.

**LLM-as-Judge Eval Layer** — Evaluates the assembled brief against a 5-criterion rubric; must score ≥ 0.75 to proceed.

**HITL Gate** — The final gate. No referral is written without explicit counselor `APPROVE_AND_LOG`.

In [ ]:
# ── Cell 5 · 7-day pipeline run ───────────────────────────────────────────────

_MODEL     = 'gemini-3.1-flash-lite'
_N_SENIORS = 10  # always batched into one signal_batch call per day

# ── Header ────────────────────────────────────────────────────────────────────
_W = 65
print('━' * _W)
print(f'  SchoolPulse · 7-Day Pipeline Run   model: {_MODEL}')
print('━' * _W)
print(f'  20 students · 7 days · {_fmt(DEMO_DATES[0])} → {_fmt(DEMO_DATES[-1])}')
print(f'  10 juniors (emoji check-ins, deterministic lookup table)')
print(f'  10 seniors (free-text check-ins, LLM-scored)')
print()
print('  Each day runs three pipeline phases then assembles a counselor brief:')
print()
print(f'  signal_batch       1 LLM call/day — all {_N_SENIORS} senior text responses batched;')
print( '                     converts free text to emotional_valence + energy_level + withdrawal_flag')
print( '  recommended_action 1 LLM call per urgent student — generates a specific,')
print( '                     counselor-ready action step (only fires when urgency is detected)')
print( '  judge_brief        1 LLM call/day — evaluates the assembled brief against a')
print( '                     5-criterion rubric; brief must score ≥ 0.75 to reach the counselor')
print()
print('━' * _W)
print()

# Auto-approve for the 7-day run; HITL scenarios are demonstrated in Cells 9-10.
def _auto_approve(brief, scorecard, session):
    return 'APPROVE_AND_LOG'

orchestrator = SchoolPulseOrchestrator(
    signal_llm=signal_llm,
    orchestrator_llm=orchestrator_llm,
    hitl_callback=_auto_approve,
)

day_results      = []
memory_snapshots = {}
total_calls      = 0

for i, date in enumerate(DEMO_DATES):
    day_checkins = [c for c in checkins if c.get('date') == date]
    result = orchestrator.run_batch(date, day_checkins, teacher_obs, registry)
    day_results.append(result)
    memory_snapshots[date] = copy.deepcopy(orchestrator.memory_store)

    reports    = result['session_state']['trend_reports']
    n_proc     = len(result['session_state']['processed_students'])
    n_urgent   = sum(1 for r in reports if r['recommended_priority'] == 'urgent')
    n_elevated = sum(1 for r in reports if r['recommended_priority'] == 'elevated')

    day_calls    = 2 + n_urgent   # signal_batch + recommended_action×N + judge_brief
    total_calls += day_calls

    calls_desc = [f'signal_batch ({_N_SENIORS} seniors)']
    if n_urgent > 0:
        calls_desc.append(f'recommended_action ×{n_urgent}')
    calls_desc.append('judge_brief')

    print(
        f'Day {i + 1} ({_short(date)}) '
        f'— {n_proc} students processed ✓   '
        f'[{n_urgent} urgent · {n_elevated} elevated]'
    )
    print(
        f'  ↳ {_MODEL}: {" · ".join(calls_desc)}'
        f'   [{day_calls} LLM calls]'
    )

    if i < len(DEMO_DATES) - 1:
        time.sleep(1)

print()
print(f'Memory Keeper: 7-day rolling window complete.  Total Gemini calls: {total_calls}')

In [ ]:
# ── Cell 6 · Student trend spotlight ─────────────────────────────────────────
# Reads Memory Keeper's accumulated state for two arc-critical students
# (S_004 and S_017) to illustrate how the 7-day rolling window works.
# For each student shows:
#   • Rolling valence baseline (average across the full 7-day window)
#   • Day 7 valence reading
#   • Whether a pattern break fired (sudden drop exceeding the 0.4 delta threshold)
#   • Consecutive low-day count
# These are the exact signals the orchestrator uses to escalate priority to urgent.
# No LLM calls — reads directly from memory_store populated by Memory Keeper.

print('Student trend spotlight — Days 1→7')
print('─' * 55)

for sid in ('S_004', 'S_017'):
    mem      = orchestrator.memory_store.get(sid, {})
    history  = mem.get('signal_history', [])
    baseline = mem.get('rolling_baseline', {}).get('avg_valence')
    consec   = mem.get('consecutive_low_days', 0)

    day7_reports = day_results[-1]['session_state']['trend_reports']
    report       = next((r for r in day7_reports if r['student_id'] == sid), {})
    priority     = report.get('recommended_priority', '?')
    pat_break    = report.get('pattern_break_detected', False)

    day7_valence = history[-1]['emotional_valence'] if history else 0.0
    baseline_str = f'{baseline:+.2f}' if baseline is not None else 'N/A'
    valence_str  = f'{day7_valence:+.2f}'

    if pat_break:
        note = f'⚠ pattern break detected  → priority: {priority}'
    elif consec >= 3:
        note = f'⚠ {consec} consecutive low days  → priority: {priority}'
    else:
        note = f'priority: {priority}'

    print(f'{sid}  baseline: {baseline_str}  →  Day 7 valence: {valence_str}  {note}')

print('─' * 55)

In [ ]:
# ── Cell 7 · Day 7 Daily Brief (formatted display) ───────────────────────────
# Renders the final counselor brief for June 28 in a bordered display box.
# The brief is structured into three sections:
#   🔴 URGENT   — students requiring immediate counselor action + LLM-generated
#                 recommended step (one call per urgent student)
#   🟡 ELEVATED — students on the watch list; rule-based action, no LLM call
#   🟢 ROUTINE  — all remaining students, no action needed today
# Key guarantee: no real student names appear — only anonymised IDs.
# Privacy Guard stripped all PII before the brief text was ever generated.

brief        = day_results[-1]['daily_brief']
date_display = _fmt(DEMO_DATES[-1])

title  = f'  SCHOOLPULSE DAILY BRIEF — {date_display}  '
border = '═' * len(title)
print(f'╔{border}╗')
print(f'║{title}║')
print(f'╚{border}╝')
print()

for line in brief.splitlines():
    if line.startswith('DAILY BRIEF') or line.startswith('Generated by:'):
        continue
    elif line.startswith('URGENT'):
        print(f'\U0001f534 {line}')
    elif line.startswith('ELEVATED'):
        print(f'\U0001f7e1 {line}')
    elif line.startswith('ROUTINE'):
        print(f'\U0001f7e2 {line}')
    else:
        print(line)

print()
print('⚠  No student names appear in this brief. Student IDs only.')
print('   Privacy Guard stripped all PII before any LLM processing.')

In [ ]:
# ── Cell 8 · LLM-as-judge scorecard ──────────────────────────────────────────
# Displays the 5-criterion rubric the judge LLM evaluated the Day 7 brief against.
# Raw scores are 0–2; displayed here normalised to 0–1.
# Weighted score: Σ (score/2 × weight) across all five criteria.
#
#   signal_coverage       0.25  — all urgent/elevated students mentioned?
#   escalation_accuracy   0.30  — urgent students correctly in the URGENT section?
#   pii_free_output       0.20  — zero real student names in the brief?  (binary)
#   counselor_action_clarity 0.15 — recommended actions specific and actionable?
#   false_positive_rate   0.10  — routine students absent from urgent/elevated?
#
# Brief must score ≥ 0.75 to be released to the counselor's HITL gate.
# If it fails, the pipeline halts and does not produce referrals.

scorecard = day_results[-1]['judge_scorecard']

criteria = [
    ('signal_coverage',          0.25),
    ('escalation_accuracy',      0.30),
    ('pii_free_output',          0.20),
    ('counselor_action_clarity', 0.15),
    ('false_positive_rate',      0.10),
]

print('LLM-as-Judge Evaluation')
print('─' * 45)
for key, _ in criteria:
    raw  = scorecard.get(key, 0)
    norm = raw / 2  # 0-2 scale -> 0-1
    tick = '✓' if norm >= 0.5 else '✗'
    print(f'  {key:<26} {norm:.2f}  {tick}')

print('─' * 45)
weighted = scorecard.get('weighted_score', 0.0)
passed   = scorecard.get('pass', False)
status   = '✅ PASSED — brief approved for counselor review' if passed else '❌ FAILED'
print(f'  Overall score: {weighted:.2f}  (threshold: 0.75)  {status}')
reason = scorecard.get('failure_reason')
if not passed and reason:
    print(f'  Failure reason: {reason}')

In [ ]:
# ── Cell 9 · Counselor HITL gate — Scenario A: APPROVE_AND_LOG ───────────────
# Shows the referral log written when the counselor selects APPROVE.
# In production this is an interactive CLI prompt; _auto_approve (Cell 5)
# has been simulating APPROVE throughout the 7-day run.
# Each urgent referral entry contains:
#   • student ID and priority
#   • recommended action (LLM-generated, cached from brief assembly)
#   • ISO timestamp of counselor approval
# Elevated students also appear in the referral log with a rule-based action.
# No referral is ever written without APPROVE_AND_LOG — _write_referrals() is
# the only call site and it lives inside the HITL approval branch.

day7          = day_results[-1]
ref_log       = day7['referral_log']
urgent_refs   = [r for r in ref_log if r['priority'] == 'urgent']

print('Counselor action: APPROVE')
print(f'Referral log written — {len(urgent_refs)} urgent entries')
for r in urgent_refs:
    sid    = r['student_id']
    pri    = r['priority']
    ts     = r['counselor_approved_at'][:16].replace('T', ' ')
    action = r.get('recommended_action', 'Counselor check-in scheduled.')
    # Trim to first sentence for display
    action_short = action.split('.')[0] + '.'
    print(f'  [{ts}] {sid} — {pri} — {action_short}')

In [ ]:
# ── Cell 10 · Counselor HITL gate — Scenarios B & C ─────────────────────────
# Demonstrates the two remaining HITL outcomes using a fresh orchestrator
# seeded with the real Day 6 memory snapshot (student history already accumulated).
#
# Scenario B — OVERRIDE_NO_ACTION
#   Counselor overrides without approving. The full pipeline runs and flags
#   urgent students, but _write_referrals() is never called → referral_log = 0.
#   Uses Fake LLMs (no API calls) — HITL blocking is independent of LLM output;
#   the student priorities come from Memory Keeper's deterministic algorithm.
#
# Scenario C — REQUEST_MORE_CONTEXT
#   Memory Keeper is queried read-only for one student's 7-day signal history.
#   No pipeline re-run and no LLM inference — data comes straight from memory_store.

from agents.llm_interface import FakeSignalLLM, FakeOrchestratorLLM

print('─' * 54)
print('Scenario B — OVERRIDE_NO_ACTION')
print('─' * 54)

override_orch = SchoolPulseOrchestrator(
    signal_llm=FakeSignalLLM(),
    orchestrator_llm=FakeOrchestratorLLM(),
    hitl_callback=lambda brief, sc, ss: 'OVERRIDE_NO_ACTION',
)
# Seed with memory state after Day 6 (before Day 7)
override_orch.memory_store = copy.deepcopy(memory_snapshots[DEMO_DATES[-2]])

day7_checkins = [c for c in checkins if c.get('date') == DEMO_DATES[-1]]
result_ov     = override_orch.run_batch(DEMO_DATES[-1], day7_checkins, teacher_obs, registry)

ov_refs  = result_ov['referral_log']
ov_audit = result_ov['audit_log']

print(f'Referral log: {len(ov_refs)} entries  ← no action taken')
if ov_audit:
    entry   = ov_audit[0]
    outcome = entry['hitl_outcome']
    ts      = entry['counselor_action_at'][:16].replace('T', ' ')
    n_urg   = entry['urgent_count']
    n_elv   = entry['elevated_count']
    print(f'Audit log:    1 entry — outcome: {outcome}  at {ts}')
    print(f'  Urgent: {n_urg} · Elevated: {n_elv} · Referrals written: 0')

print()

# ── Scenario C: REQUEST_MORE_CONTEXT ─────────────────────────────────────────
# Memory Keeper is re-invoked read-only to surface a student's 7-day history.
# Privacy Guard and Signal Detector are NOT re-called — no new LLM inference.

print('─' * 54)
print('Scenario C — REQUEST_MORE_CONTEXT (S_004)')
print('─' * 54)
print('Memory Keeper read-only invocation (no pipeline re-run):')
print()

history_summary = orchestrator.get_student_history_summary('S_004')
print(history_summary)
print()
print('→ Counselor reviews 7-day trend, then proceeds with APPROVE_AND_LOG.')

## What SchoolPulse demonstrated

| Concept | Implementation |
|---|---|
| Multi-agent architecture | Orchestrator coordinating 3 sub-agents |
| Agent Skills (SKILL.md) | Three skills with EDD eval cases |
| MCP layer | `mcp_server.py` — runnable stdio MCP server (FastMCP) with 4 tools; `mcp_config.json` wires local stdio (demo) and `@modelcontextprotocol/server-gdrive` (production Google Sheets); follows whitepaper *consumption-over-creation* principle |
| Security & Privacy | Privacy Guard + PII stripping before any LLM context |
| LLM-as-judge | Automated brief evaluation (threshold: 0.75) |
| Human-in-the-loop | Counselor approval required — no auto-referrals |
| Spec-Driven Development | SPEC.md written before any code |

**The system surfaces patterns. The counselor makes the call.**

[GitHub](https://github.com/ritikaAllen/school-pulse) · [SPEC.md](../SPEC.md)